In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="0"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="0"

import json
from tqdm import tqdm
from typing import Dict, List, Any

import torch
import torch_npu

from datasets import load_dataset
from transformers import LlamaForCausalLM, AutoTokenizer

/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
task = "humaneval"
task_params = {
    "humaneval": {
        "path": "evalplus/humanevalplus",
        "split": "test"
    },
    "magicoder": {
        "path": "/data/datasets/Magicoder-Evol-Instruct-110K",
        "split": "train"
    }
}
path = task_params[task]['path']
split = task_params[task]['split']
# dataset = load_dataset("/data/datasets/Magicoder-Evol-Instruct-110K")["train"]
dataset = load_dataset(path)[split]

base_model_path = "/data/pretrained-models/meta/Llama-3.2-1B-Instruct"
main_model_path = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
model_1b = LlamaForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.bfloat16,
    device_map="npu:0")
model_3b = LlamaForCausalLM.from_pretrained(
    main_model_path,
    # "/data/lihz/projects/instruct/IFT/3b-e3",
    # "/data/pretrained-models/meta/Llama-3.1-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="npu:0")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
def preprocess(example:Dict[str, str])->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which serves as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
            "entry_point": example['entry_point']
        }
    elif "magicoder":
        return {
            "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
            "resp": f"{example['response']}<|eot_id|>",
            "entry_point": example['entry_point']
        }
def postprocess(example:Dict[str, str])->Dict[str, List[int]]:
    inst = tokenizer.encode(example["inst"], add_special_tokens=False)
    if task == "humaneval":
        return {
            "input_ids": inst,
            "entry_point": example['entry_point']
        }
    elif task == "magicoder":
        resp = tokenizer.encode(example["resp"], add_special_tokens=False)
        return {
            "input_ids": inst + resp,
            "labels": [-100] * len(inst) + resp,
            "entry_point": example['entry_point']
        }

In [4]:
tag = 10
dataset = dataset.map(
    preprocess,
    num_proc=4,
    remove_columns=dataset.column_names,
    # load_from_cache_file=False,
)
# print(dataset[tag]['inst']+dataset[tag]['resp'])
dataset = dataset.map(
    postprocess,
    num_proc=4,
    remove_columns=dataset.column_names,
    # load_from_cache_file=False,
)
print(tokenizer.decode(dataset[tag]['input_ids']))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
def is_palindrome(string: str) -> bool:
    """ Test if given string is a palindrome """
    return string == string[::-1]


def make_palindrome(string: str) -> str:
    """ Find the shortest palindrome that begins with a supplied string.
    Algorithm idea is simple:
    - Find the longest postfix of supplied string that is a palindrome.
    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.
    >>> make_palindrome('')
    ''
    >>> make_palindrome('cat')
    'catac'
    >>> make_palindrome('cata')
    'catac'
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the prob

In [5]:
with torch.no_grad():
    input_kwargs = {
        "input_ids": torch.tensor([dataset[tag]["input_ids"]], device=model_1b.device),
        "do_sample": False,
        "use_cache": True,
        "max_new_tokens": 512,
    }
    seq_1b = model_1b.generate(**input_kwargs)
    seq_3b = model_3b.generate(**input_kwargs)

/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as

In [6]:
print(tokenizer.batch_decode(seq_1b)[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
def is_palindrome(string: str) -> bool:
    """ Test if given string is a palindrome """
    return string == string[::-1]


def make_palindrome(string: str) -> str:
    """ Find the shortest palindrome that begins with a supplied string.
    Algorithm idea is simple:
    - Find the longest postfix of supplied string that is a palindrome.
    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.
    >>> make_palindrome('')
    ''
    >>> make_palindrome('cat')
    'catac'
    >>> make_palindrome('cata')
    'catac'
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the prob

In [7]:
print(tokenizer.batch_decode(seq_3b)[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
def is_palindrome(string: str) -> bool:
    """ Test if given string is a palindrome """
    return string == string[::-1]


def make_palindrome(string: str) -> str:
    """ Find the shortest palindrome that begins with a supplied string.
    Algorithm idea is simple:
    - Find the longest postfix of supplied string that is a palindrome.
    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.
    >>> make_palindrome('')
    ''
    >>> make_palindrome('cat')
    'catac'
    >>> make_palindrome('cata')
    'catac'
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the prob

In [32]:
alpha = 0.1
do_sample = False
max_new_tokens = 512
input_ids = torch.tensor([dataset[tag]["input_ids"]])
# print(dataset[tag]["input_ids"])
# with torch.no_grad():
with torch.inference_mode():
    input_kwargs_1b = {
        "input_ids": input_ids.to(model_1b.device),
        "use_cache": True,
        "past_key_values": None,
        "return_dict": True,
    }
    input_kwargs_3b = {
        "input_ids": input_ids.to(model_3b.device),
        "use_cache": True,
        "past_key_values": None,
        "return_dict": True,
    }
    for i in range(max_new_tokens):
        outputs_1b = model_1b(**input_kwargs_1b)
        outputs_3b = model_3b(**input_kwargs_3b)
        logits_1b = outputs_1b.logits[:, -1:, :]
        logits_3b = outputs_3b.logits[:, -1:, :]
        probs_1b = logits_1b.softmax(dim=-1)
        probs_3b = logits_3b.softmax(dim=-1)
        pkv_1b = outputs_1b.past_key_values
        pkv_3b = outputs_3b.past_key_values
        
        log_prob_1b = logits_1b.log_softmax(dim=-1)
        log_prob_3b = logits_3b.log_softmax(dim=-1)
        log_prob = log_prob_3b - log_prob_1b
        log_prob = torch.where(
            probs_3b < (alpha * torch.max(probs_3b, dim=-1, keepdim=True).values),
            -torch.finfo(log_prob.dtype).max,
            log_prob
            )
        if do_sample:
            prob = torch.softmax(log_prob, dim=-1)
            bsz, seq_len, _ = prob.size()
            next_token = torch.multinomial(prob.view(bsz * seq_len, -1), num_samples=1).view(bsz, -1)
        else:
            next_token = torch.argmax(log_prob, dim=-1)
        input_kwargs_1b = {
            "input_ids": next_token.to(model_1b.device),
            "use_cache": True,
            "past_key_values": pkv_1b,
            "return_dict": True,
        }
        input_kwargs_3b = {
            "input_ids": next_token.to(model_3b.device),
            "use_cache": True,
            "past_key_values": pkv_3b,
            "return_dict": True,
        }
        input_ids = torch.cat([input_ids, next_token.to(input_ids.device)], dim=-1)
        if next_token.eq(tokenizer.eos_token_id).sum() == 1:
            break

In [33]:
# print(input_ids)
print(tokenizer.batch_decode(input_ids)[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
def is_palindrome(string: str) -> bool:
    """ Test if given string is a palindrome """
    return string == string[::-1]


def make_palindrome(string: str) -> str:
    """ Find the shortest palindrome that begins with a supplied string.
    Algorithm idea is simple:
    - Find the longest postfix of supplied string that is a palindrome.
    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.
    >>> make_palindrome('')
    ''
    >>> make_palindrome('cat')
    'catac'
    >>> make_palindrome('cata')
    'catac'
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the prob

: 

In [ ]:
results = []
eos_tokens = [
    "<|endoftext|>",
    "<|endofmask|>",
    "</s>",
    "\nif __name__",
    "\ndef main(",
    "\nprint(",
    "<|eot_id|>"
]
target_path = os.path.basename(base_model_path) + os.path.basename(main_model_path) + task + ".jsonl"
for i, item in enumerate(tqdm(dataset)):
    input_ids = torch.tensor([item["input_ids"]])
    with torch.inference_mode():
        input_kwargs_1b = {
            "input_ids": input_ids.to(model_1b.device),
            "use_cache": True,
            "past_key_values": None,
            "return_dict": True,
        }
        input_kwargs_3b = {
            "input_ids": input_ids.to(model_3b.device),
            "use_cache": True,
            "past_key_values": None,
            "return_dict": True,
        }
        for i in range(max_new_tokens):
            outputs_1b = model_1b(**input_kwargs_1b)
            outputs_3b = model_3b(**input_kwargs_3b)
            logits_1b = outputs_1b.logits[:, -1:, :]
            logits_3b = outputs_3b.logits[:, -1:, :]
            probs_1b = logits_1b.softmax(dim=-1)
            probs_3b = logits_3b.softmax(dim=-1)
            pkv_1b = outputs_1b.past_key_values
            pkv_3b = outputs_3b.past_key_values
            
            log_prob_1b = logits_1b.log_softmax(dim=-1)
            log_prob_3b = logits_3b.log_softmax(dim=-1)
            log_prob = log_prob_3b - log_prob_1b
            log_prob = torch.where(
                probs_3b < (alpha * torch.max(probs_3b, dim=-1, keepdim=True).values),
                -torch.finfo(log_prob.dtype).max,
                log_prob
                )
            if do_sample:
                prob = torch.softmax(log_prob, dim=-1)
                bsz, seq_len, _ = prob.size()
                next_token = torch.multinomial(prob.view(bsz * seq_len, -1), num_samples=1).view(bsz, -1)
            else:
                next_token = torch.argmax(log_prob, dim=-1)
            input_kwargs_1b = {
                "input_ids": next_token.to(model_1b.device),
                "use_cache": True,
                "past_key_values": pkv_1b,
                "return_dict": True,
            }
            input_kwargs_3b = {
                "input_ids": next_token.to(model_3b.device),
                "use_cache": True,
                "past_key_values": pkv_3b,
                "return_dict": True,
            }
            input_ids = torch.cat([input_ids, next_token.to(input_ids.device)], dim=-1)
            if next_token.eq(tokenizer.eos_token_id).sum() == 1:
                break
        
        outputs = tokenizer.batch_decode(input_ids[:, len(item["input_ids"]):])
        for output in outputs:
            min_index = 10000
            for eos in eos_tokens:
                if eos in output:
                    min_index = min(min_index, output.index(eos))
            results.append(output[:min_index].replace("\t", "    "))

from evalplus.sanitize import sanitize
with open(target_path, "a") as f:
    for i, (impl, item) in enumerate(zip(results, dataset)):
        sanitized_solution = sanitize(
            impl, entrypoint=item['entry_point']
        )
        f.write(
            json.dumps(
                {"task_id": f"HumanEval/{i}", "solution": sanitized_solution}
            )
            + "\n"
        )